# Smart Student Performance Predictor - original step-by-step notebook

This notebook is the simple baseline walkthrough (Logistic Regression vs Decision Tree). Run it from the `notebooks/` folder (paths use `../`). The full upgraded workflow (4 tuned models, ROC-AUC, Shapley explanations, fairness) is in `src/model_training.py` and is run with `python train_model.py`. Install Jupyter separately: `pip install notebook`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Load and inspect the data

In [ ]:
df = pd.read_csv("../data/student_performance.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())
df.describe()

## 2. Clean the data
Remove duplicates and impossible values. Missing feature values are filled **later inside the Pipeline** (after the split) to avoid data leakage.

In [ ]:
df = df.drop_duplicates().dropna(subset=["Result"]).copy()
for c in ["Attendance", "Previous_Score", "Assignment_Score"]:
    df[c] = df[c].clip(0, 100)
df["Result"].value_counts()

## 3. Exploratory data analysis

In [ ]:
num = ["Age","Study_Hours","Attendance","Previous_Score","Assignment_Score","Sleep_Hours"]
cat = ["Gender","Participation","Internet_Access","Extracurricular"]
sns.countplot(data=df, x="Result"); plt.title("Class distribution"); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, c in zip(axes.ravel(), num):
    sns.boxplot(data=df, x="Result", y=c, ax=ax); ax.set_title(f"{c} vs Result")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, c in zip(axes, cat):
    (df.groupby(c)["Result"].apply(lambda s: (s=="Pass").mean()*100)).plot(kind="bar", ax=ax)
    ax.set_title(f"Pass % by {c}")
plt.tight_layout(); plt.show()

In [ ]:
tmp = df[num].copy(); tmp["Passed"] = (df["Result"]=="Pass").astype(int)
sns.heatmap(tmp.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0); plt.show()

**What we see:** Pass students tend to have higher study hours, attendance, previous and assignment scores, and higher class participation.

## 4. Split first, then preprocess
The Pipeline learns imputation values, scaling and encoding **only from training data**.

In [ ]:
X = df[num + cat]
y = (df["Result"] == "Pass").astype(int)     # Pass=1, Fail=0
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

def make_pipeline(model):
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat)])
    return Pipeline([("preprocess", prep), ("model", model)])

## 5. Train two models

In [ ]:
models = {"Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
          "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)}
fitted, cv = {}, {}
for name, m in models.items():
    p = make_pipeline(m)
    cv[name] = cross_val_score(p, X_train, y_train, cv=5, scoring="f1").mean()
    fitted[name] = p.fit(X_train, y_train)
cv

## 6. Evaluate on the unseen test set

In [ ]:
rows = []
for name, p in fitted.items():
    pred = p.predict(X_test)
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred), "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred)})
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Fail","Pass"]).plot(cmap="Blues")
    plt.title(name); plt.show()
pd.DataFrame(rows).round(3)

## 7. Select the best model and save it
Selection uses cross-validation F1 on the **training** data, so the test set stays a fair final exam.

In [ ]:
best = max(cv, key=cv.get); print("Best model:", best)
joblib.dump(fitted[best], "../models/student_performance_model.pkl")

## 8. Predict for a new student

In [ ]:
model = joblib.load("../models/student_performance_model.pkl")
new = pd.DataFrame([{"Age":20,"Gender":"Female","Study_Hours":3,"Attendance":70,"Previous_Score":55,
                     "Assignment_Score":58,"Sleep_Hours":6,"Participation":"Medium",
                     "Internet_Access":"Yes","Extracurricular":"Yes"}])
print("Prediction:", "PASS" if model.predict(new)[0]==1 else "FAIL")
print("P(pass):", round(model.predict_proba(new)[0][1], 3))